# Summary

01_quickstart.ipynb

Single scan inference end-to-end. Load a scan, run NeuroFM.predict(), show the output brain health and latent features.

# Install

Unfortunately due to dependency conflicts between Tensorflow 2.13 and ipython, installation needs to take place outside the notebook in your ipykernel environment (unless you're running a kernel from the NeuroFM docker container, then you can skip this step).

Please run the following in your terminal Python environment before starting the kernel (mamba, etc.) to create and activate a dedicated environment, then install dependencies:
```bash
pip install "neurofm[notebooks] @ git+https://github.com/rockNroll87q/NeuroFM.git"
pip install templateflow==24.1.0 "numpy<=1.24.3" "typing-extensions<4.6.0"
```

**Note:** The `typing-extensions` pin is required for TensorFlow compatibility. Installing inside the notebook is not recommended as it conflicts with ipython's own dependencies. 


If you run into issues, we recommend using the NeuroFM notebook docker container `rocknroll87q/neurofm:notebook`:

Run: `docker run --rm -p 8888:8888 rocknroll87q/neurofm:notebook`.

Open your browser to `http://localhost:8888`.

# Python API

## Imports

In [ ]:
# Initial import may take ~30s due to internal TensorFlow import.
import templateflow.api as tflow

from neurofm import NeuroFM
from neurofm.model import BRAIN_HEALTH_KEYS


## Create template test vol

For the purposes of illustration, we load an MNI test volume. In your own use-case, this can be replaced by a single volume, a directory of volumes, or a `.csv` file with an input column pointing to your volume files.

In [2]:
print("Fetching MNI152NLin2009cAsym 1mm T1w template from TemplateFlow...")
template_path = tflow.get(
    "MNI152NLin2009cAsym",
    resolution=1,
    desc="brain",
    suffix="T1w",
    extension=".nii.gz",
)

if template_path is None:
    print("TemplateFlow could not fetch the template. Check your internet connection.")

Fetching MNI152NLin2009cAsym 1mm T1w template from TemplateFlow...


## Run NeuroFM

In [3]:
model_variant = "neurofm-s" # choose your variant
device = "cpu"

In [ ]:
model = NeuroFM(
    variant=model_variant,
    device=device,
)

In [5]:
results = model.predict(template_path, outputs=['brain_health', 'latent'])
brain_health = results['brain_health']
latent = results['latent']

2026-03-26 15:53:34.229 | DEBUG    | neurofm.io:_reorient:163 - Reorienting from 'RAS' to 'LIA'.
2026-03-26 15:53:34.269 | DEBUG    | neurofm.io:_resample:186 - Resampling: shape (193, 193, 229) -> (256, 256, 256), zooms (1.0, 1.0, 1.0) -> (1.0, 1.0, 1.0).


### Brain health outputs

In [6]:
print(f'Output order: {BRAIN_HEALTH_KEYS}')

Output order: ['brain_age', 'sex', 'ventricle_volume', 'brain_volume']


In [7]:
print(f'Predicted age: {brain_health[0]} years')
print(f'Predicted sex: {brain_health[1]} (0 == F, 1 == M)')
print(f'Predicted ventricle vol: {brain_health[2]} mm^3') 
print(f'Predicted brain vol: {brain_health[3]} mm^3')

Predicted age: 58.325374603271484 years
Predicted sex: 0.0 (0 == F, 1 == M)
Predicted ventricle vol: 23985.220703125 mm^3
Predicted brain vol: 1695925.625 mm^3


### Latent outputs

Latent embedded representation of the input brain volume. Dimensionality depends on model variant.

In [8]:
print(latent)

[ 0.03749827  0.32106206 -0.18453555 -0.2014856   0.10724859 -0.21201599
  0.08957316  0.41148922  0.43628138 -0.13389575 -0.16290456 -0.03286832
  0.12378959  0.27158847 -0.0074301  -0.08271644  0.1656631   0.26775104
  0.08607028 -0.21734792  0.04916124  0.02662318 -0.0926581   0.00638308
  0.34374702 -0.08816021  0.24651676 -0.1646723  -0.12482292  0.13885516
  0.4952996   0.29678977  0.33648756  0.08423754  0.06424014  0.4397002
  0.404352    0.36315352  0.09246024  0.09763176 -0.04595795  0.35724372
 -0.17047855  0.14726704  0.22023746  0.4213118   0.48308653  0.35445228
  0.2983006   0.19760326  0.0497797   0.09764866  0.17741814  0.479736
 -0.19878145  0.44830608 -0.05329982  0.43187433 -0.12600093  0.15168023
  0.24960598 -0.18141529  0.33493    -0.00737642  0.11763827  0.2577043
 -0.11370381  0.27564722  0.32710552  0.15555425  0.26587585  0.18161464
  0.38085088  0.30601293  0.00314309 -0.10060852  0.16663846  0.19807334
  0.32073498  0.09081528  0.35595882 -0.17386453  0.106

# Script method

## Create template test vol

Create the template test data and save to /tmp/

In [ ]:
!python ../scripts/get_test_data.py -o /tmp/test_data/ --mode template -n 1

## Run inference script to produce output files

In [ ]:
!python ../scripts/run_inference.py -i /tmp/test_data/MNI152_test.nii.gz \
    -o /tmp/nfm_results --device cpu --model neurofm-s --outputs brain_health,latent

In [13]:
!ls /tmp/nfm_results

individuals                 latent_embeddings_index.csv
latent_embeddings.npy       results_summary.csv


Show the output summary .csv (brain health data for all volumes)

In [14]:
import pandas as pd

pd.read_csv('/tmp/nfm_results/results_summary.csv')

,input,brain_age,sex,ventricle_volume,brain_volume
0,/tmp/test_data/MNI152_test.nii.gz,58.325359,0.0,23985.238281,1695925.25
